# Step By Step 07 - 风险因子模型

Construct a factor risk model, optimize weights and attribute active returns.


In [ ]:
import datetime as dt
import numpy as np
import pyarrow as pa
from alphamind.analysis.perfanalysis import perf_attribution_by_pos
from alphamind.portfolio.meanvariancebuilder import mean_variance_builder
from alphamind.portfolio.riskmodel import FactorRiskModel

codes = ["A", "B", "C"]
covariance = pa.table({
    "Factor": ["BETA", "SIZE"],
    "BETA": [0.02, 0.003],
    "SIZE": [0.003, 0.01],
})
exposure = pa.table({
    "code": codes,
    "BETA": [1.0, 0.8, 1.2],
    "SIZE": [0.2, 0.4, -0.1],
})
specific = pa.table({"code": codes, "idsync": [0.01, 0.015, 0.012]})
model = FactorRiskModel(covariance, exposure, specific)
profile = model.get_risk_profile(codes)
benchmark = np.full(3, 1.0 / 3.0)
status, objective, weights = mean_variance_builder(
    np.array([0.02, 0.03, 0.01]),
    profile,
    benchmark,
    0.0,
    1.0,
    np.ones((3, 1)),
    (np.array([1.0]), np.array([1.0])),
)
assert status in ("optimal", "optimal_inaccurate")
trade_date = dt.datetime(2026, 1, 5)
active = pa.table({"trade_date": [trade_date] * 3, "code": codes, "weight": weights - benchmark})
returns = pa.table({"trade_date": [trade_date] * 3, "code": codes, "return": [0.01, -0.02, 0.015]})
risks = pa.table({
    "trade_date": [trade_date] * 3,
    "code": codes,
    "BETA": exposure["BETA"],
    "SIZE": exposure["SIZE"],
})
attribution = perf_attribution_by_pos(active, returns, risks)
assert attribution.num_rows == 1
attribution
